In [ ]:
import gzip
import pyshark

# Step 1: Decompress the .pcap.gz file
def decompress_gz(gz_file, output_file):
    with gzip.open(gz_file, 'rb') as f_in:
        with open(output_file, 'wb') as f_out:
            f_out.write(f_in.read())

# Step 2: Analyze the decompressed PCAP file
def analyze_pcap(pcap_file):
    # Read the pcap file using pyshark
    cap = pyshark.FileCapture(pcap_file)
    
    # Display general packet statistics
    print(f"Total packets: {len(cap)}")
    
    # Example: Analyze first few packets
    for packet in cap[:10]:  # Get the first 10 packets
        print(packet)  # Print packet details
    
    # Filter specific protocols (e.g., HTTP, DNS, TCP, etc.)
    http_packets = [pkt for pkt in cap if 'HTTP' in pkt]
    print(f"Total HTTP packets: {len(http_packets)}")
    
    # Close the capture after use
    cap.close()

# Main execution
if __name__ == "__main__":
    gz_file = '/home/shyam/jupy/ddos_scrubber/data/amppot_dataset/amppot-ynu_sensor009_20210609.pcap.gz'
    pcap_file = '/home/shyam/jupy/ddos_scrubber/data/amppot_dataset/amppot-ynu_sensor009_20210609.pcap'
    
    # Step 1: Decompress
    decompress_gz(gz_file, pcap_file)
    
    # Step 2: Analyze the PCAP
#     analyze_pcap(pcap_file)


In [15]:
# Read amppot file
import pandas as pd
filename = "/home/shyam/phd-work/DDOS/amppot-data/amppot-2024-11-01.csv.gz"
# Define custom headers
column_names = ['hpdate', 'dport', 'src', 't_start', 't_end', 'packets', 'hpids']

# Read the CSV file without headers and assign column names
df = pd.read_csv(filename, compression='gzip', header=None, names=column_names)

# Convert to datetime
df['t_start'] = pd.to_datetime(df['t_start'])
df['t_end'] = pd.to_datetime(df['t_end'])
df['hpdate'] = pd.to_datetime(df['hpdate']).dt.date

# Filter: duration > 5 minutes AND hpdate == 2024-12-03
filtered_df = df[
    (df['t_end'] - df['t_start'] > pd.Timedelta(minutes=5)) &
    (df['hpdate'] == pd.to_datetime('2024-11-03').date()) &
#     (df['hpdate'] >= pd.to_datetime('2024-12-01').date()) &
    (df['t_end'] - df['t_start'] < pd.Timedelta(minutes=60))
]

# Show result
source_ips = filtered_df["src"]
print(f"Number of attacks {len(source_ips)} and number of unique victim IPs  {len(source_ips.unique())}")

Number of attacks 1867 and number of unique victim IPs  1375


In [3]:
filtered_df

,hpdate,dport,src,t_start,t_end,packets,hpids
14565,2024-11-02,19,1.145.191.244,2024-11-02 11:22:28,2024-11-02 11:33:02,1805,4
14566,2024-11-02,19,1.248.220.170,2024-11-02 13:59:00,2024-11-02 14:22:58,1143,4
14570,2024-11-02,19,2.90.223.71,2024-11-02 07:48:54,2024-11-02 07:54:13,525,4
14571,2024-11-02,19,2.97.21.138,2024-11-02 00:39:20,2024-11-02 01:16:29,1942,4
14572,2024-11-02,19,2.108.223.69,2024-11-02 21:11:51,2024-11-02 21:16:59,1827,4
...,...,...,...,...,...,...,...
26205,2024-11-02,123,222.175.222.24,2024-11-02 08:56:35,2024-11-02 09:31:19,4111,6
26233,2024-11-02,1900,114.132.230.96,2024-11-02 11:19:32,2024-11-02 11:27:10,2048,8
26234,2024-11-02,1900,118.25.26.201,2024-11-02 07:23:42,2024-11-02 08:04:03,1763,8
26249,2024-11-02,11211,185.211.78.135,2024-11-02 17:54:59,2024-11-02 18:00:34,4650,2


In [14]:
# Check BGP updates (announcements) for a few targeted IP addresses in amppot file
import pybgpstream
import datetime
from datetime import datetime, timedelta, timezone
import csv

ips = filtered_df["src"]
start_times = filtered_df["t_start"]
end_times = filtered_df["t_end"]

for idx, ip in enumerate(ips[0:4], start = 0):
    prefix = str(ip) +"/32"

    # Convert to datetime object
    # Parse the string as a datetime object with UTC timezone (GMT == UTC)
    start_dt = datetime.strptime(str(start_times.iloc[idx]), "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    posix_start_time = int(start_dt.timestamp())
    
    end_dt = datetime.strptime(str(end_times.iloc[idx]), "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    posix_end_time = int(end_dt.timestamp())
    
    from_time =  posix_start_time - 1800 # - 600 for 10 minutes, -7200 for 2 hours early
    until_time = posix_end_time + 1800 # +600 for 10 minutes, + 7200 for 2 hours later

    # Convert to GMT time
    from_time_utc = datetime.utcfromtimestamp(from_time)
    until_time_utc = datetime.utcfromtimestamp(until_time)

    # Convert UTC datetime to string
    from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
    until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

#     print("Time %s and %s" %(from_time_utc_str, until_time_utc_str))

    stream = pybgpstream.BGPStream(
        from_time=from_time_utc_str, 
        until_time=until_time_utc_str,
        record_type="updates", # By default it is Update announcement
        project="ris",
#         project="routeviews-stream",
        filter="prefix less "+prefix
        )
    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    p = [] # List containing immediate provider
    o = [] # Origin ASN

#     print("Extracting records for prefix %s" %prefix)


    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        for elem in rec:
            # Find second ASN in an AS path
            if elem.type == "A":
                as_path = elem.fields["as-path"]
                pfx = elem.fields["prefix"]

                # Convert as_path into list
                as_path_list = as_path.split()
                orig = as_path_list[-1]
            else:
                as_path_list = []
                orig = []
                prefix = "" # For some updates with had error state e.g. U|S| instead of A or W
                pfx = ""
            # List of 25 confirmed scrubbers from bgp.tools ddosm tags and DDoScovery paper.
            scrubbers = ['32787', '13335', '19551', '19905', '198949', '57724', '54113', '21859', '19324', 
                         '3223', '137409', '396998', '30456', '8757', '34309', '35280', '197068', 
                         '199524', '45474', '200020', '42649', '59796', '20052', ' 394009', '401073'] 

            single_data = []
            # Discard different forms of origins example AS-Set, confederation set/sequence. 
            # Take only a single AS origin which is common for a scrubbing activity
            matched_scrubber = set(scrubbers) & set(as_path_list)
            
            # Convert it into string
            scrubber = ','.join(matched_scrubber)
                                    
            if pfx != '0.0.0.0/0' and pfx!= '' and len(as_path_list) > 1 and scrubber:  # intersection is not empty
                
                # Find second ASN in an AS path
                second_as = as_path_list[-2]
                # Extract its upstream provider to check if that one contains an scrubbers' ASN
                single_data.append({"prefix": pfx})
                single_data.append({"provider": second_as})
                single_data.append({"origin": orig})
                single_data.append({"time": time})
                single_data.append({"as_path": as_path_list})
                single_data.append({"scrubber": scrubber})
                single_data.append({"t_start": posix_start_time})
                single_data.append({"t_end": posix_end_time})
                p.append(single_data)
    
    providers = [entry[1]['provider'] for entry in p]
    # Find number of peers that are listed as DDoS mitigation 
    matching_elements = set(providers) & set(scrubbers)
#     print("Total matching ASes ",len(matching_elements))
    if (scrubber) and len(p) >0:
        # Save results into a csv file.
        # Flatten each list of single-key dictionaries into a single dictionary
        flattened_data = [{k: v for d in entry for k, v in d.items()} for entry in p]
        # Write to CSV
        with open('output_'+str(idx)+'.csv', 'a', newline='') as csvfile:
            fieldnames = ['prefix', 'provider', 'origin', 'time', 'as_path', 'scrubber', 't_start', 't_end']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for row in flattened_data:
                writer.writerow(row)
        print("Written to a file: output_"+str(idx)+".csv")
    print("Done for prefix %s" %prefix)
print("Completed.")

Done for prefix 


WARN: INVALID_MSG: Unexpected ADD-PATH OPEN Capability length (8), expecting 4 bytes (parsebgp_bgp_open.c:86)


KeyboardInterrupt: 

In [6]:
ips = filtered_df["src"]
print(ips)

14565     1.145.191.244
14566     1.248.220.170
14570       2.90.223.71
14571       2.97.21.138
14572      2.108.223.69
              ...      
26205    222.175.222.24
26233    114.132.230.96
26234     118.25.26.201
26249    185.211.78.135
26253    185.211.78.181
Name: src, Length: 1543, dtype: object


In [39]:
# Merge all those output_<index>.csv files into a single file.
import pandas as pd
import glob

# Find all CSV files matching the pattern
csv_files = glob.glob("output_*.csv")

# Read and concatenate all files
df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV file
merged_df.to_csv("../data/merged_output_04_dec_2024.csv", index=False)
            

In [7]:
# Cases where a scrubber appears as a provider
import pandas as pd
merged_df = pd.read_csv("../data/merged_output_03_dec_2024.csv")
merged_df["time"] = merged_df['time'].astype(int) # Modify
merged_df["provider"] = merged_df["provider"].astype("string")
merged_df["origin"] = merged_df["origin"].astype("string")
scrubber_provider = merged_df[(merged_df["scrubber"] == merged_df["provider"]) & 
                             (merged_df["prefix"].str.contains("/24"))]
print(f"No. of prefixes using a scrubber as an immediate peer {len(scrubber_provider['prefix'].unique())}")

# Cases where a scrubber appears as an origin
scrubber_origin = merged_df[merged_df["scrubber"] == merged_df["origin"]]
print(f"No. of prefixes using a scrubber as an origin {len(scrubber_origin['prefix'].unique())}")

# a[a["prefix"].str.contains("/24")].iloc[5:40]

No. of prefixes using a scrubber as an immediate peer 755
No. of prefixes using a scrubber as an origin 6


In [5]:
scrubber_provider["scrubber"].unique()

array(['13335', '34309', '396998', '137409', '199524'], dtype=object)

In [8]:
# a = scrubber_provider[scrubber_provider["prefix"].str.contains("/24")]
# a.sort_values(by='time')
# a["scrubber"].unique()
scrubber_provider[(scrubber_provider["provider"] == "13335")
#                  & (scrubber_provider["prefix"] == "193.46.81.0/24")
                 ]

,prefix,provider,origin,time,as_path,scrubber,t_start,t_end
53,185.211.78.0/24,13335,32708,1733253212,"['2500', '13335', '32708']",13335,1733253583,1733253897
54,185.211.78.0/24,13335,32708,1733253212,"['151381', '13335', '32708']",13335,1733253583,1733253897
55,185.211.78.0/24,13335,32708,1733253272,"['25152', '13335', '32708']",13335,1733253583,1733253897
56,185.211.78.0/24,13335,32708,1733253285,"['4777', '13335', '32708']",13335,1733253583,1733253897
57,185.211.78.0/24,13335,32708,1733253914,"['37721', '13335', '32708']",13335,1733253583,1733253897
913,185.211.78.0/24,13335,32708,1733226255,"['37721', '13335', '32708']",13335,1733226518,1733226848
914,185.211.78.0/24,13335,32708,1733226357,"['37721', '13335', '32708']",13335,1733226518,1733226848
915,185.211.78.0/24,13335,32708,1733227003,"['24482', '13335', '32708']",13335,1733226518,1733226848
916,185.211.78.0/24,13335,32708,1733227004,"['34800', '58057', '50673', '13335', '32708']",13335,1733226518,1733226848
917,185.211.78.0/24,13335,32708,1733227004,"['12859', '13335', '32708']",13335,1733226518,1733226848


In [1]:
# Prefixes which were seen on route collectors after the time they were detected by AmpPot.
after_seen_on_amppot = scrubber_provider[(scrubber_provider["provider"] == "13335") & 
                                         (scrubber_provider["time"] > scrubber_provider["t_start"])
                                        ]
#  
a = scrubber_provider[
    (scrubber_provider["provider"] == "59796") & 
    (scrubber_provider["time"] > scrubber_provider["t_start"]) & 
    (scrubber_provider["time"] < scrubber_provider["t_end"])  &
    ((scrubber_provider["t_end"].astype(float) - scrubber_provider["t_start"].astype(float)) > 300)
]



NameError: name 'scrubber_provider' is not defined

In [1]:
# Python code that takes input as date(yyyy/mm/dd/) and ASN from a user
# Fetch ROA records from https://ftp.ripe.net/rpki/ripencc.tal/yyyy/mm/dd/roas.csv.xz. 
# It has fields: fields: URI, ASN, IP Prefix, Max Length, Not Before, Not After
# Show prefixes with that ASN
import lzma
import csv
import requests
from io import BytesIO
from datetime import datetime

def fetch_roas(date_str: str, asn_input: str):
    try:
        # Validate and parse date
        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
        url_date = date_obj.strftime('%Y/%m/%d')
        url = f'https://ftp.ripe.net/rpki/ripencc.tal/{url_date}/roas.csv.xz'
        
        print(f"Fetching ROA data from: {url}")
        response = requests.get(url)
        response.raise_for_status()

        # Decompress .xz content
        with lzma.open(BytesIO(response.content)) as f:
            csv_reader = csv.DictReader(f.read().decode().splitlines())
            
            print(f"ROA Records for ASN {asn_input} on {date_str}:\n")
            found = False
            for row in csv_reader:
                if row['ASN'].strip().upper() == asn_input.upper():
                    print(f"{row['IP Prefix']} (Max Length: {row['Max Length']})")
                    found = True

            if not found:
                print("No records found for that ASN.")
    except Exception as e:
        print(f"Error: {e}")

# Example usage
if __name__ == "__main__":
    date_input = input("Enter date (yyyy/mm/dd): ").strip()
    asn_input = input("Enter ASN (e.g., AS12345): ").strip()
    fetch_roas(date_input, asn_input)


Enter date (yyyy/mm/dd): 2024/06/11
Enter ASN (e.g., AS12345): AS198949
Fetching ROA data from: https://ftp.ripe.net/rpki/ripencc.tal/2024/06/11/roas.csv.xz
ROA Records for ASN AS198949 on 2024/06/11:

5.57.96.0/19 (Max Length: 24)
5.100.144.0/21 (Max Length: 21)
37.64.0.0/13 (Max Length: 24)
45.92.12.0/24 (Max Length: 24)
45.92.13.0/24 (Max Length: 24)
45.92.14.0/24 (Max Length: 24)
45.92.15.0/24 (Max Length: 24)
46.35.0.0/19 (Max Length: 24)
46.165.64.0/18 (Max Length: 24)
46.218.0.0/16 (Max Length: 24)
62.8.0.0/19 (Max Length: 24)
62.39.0.0/16 (Max Length: 24)
62.62.169.0/24 (Max Length: 24)
62.62.128.0/17 (Max Length: 24)
62.85.128.0/19 (Max Length: 24)
62.90.135.0/24 (Max Length: 24)
62.106.128.0/17 (Max Length: 24)
62.129.160.0/19 (Max Length: 24)
62.241.64.0/18 (Max Length: 24)
77.84.0.0/16 (Max Length: 24)
77.95.81.0/24 (Max Length: 24)
77.104.0.0/18 (Max Length: 24)
77.128.0.0/13 (Max Length: 24)
77.136.0.0/16 (Max Length: 24)
77.137.196.0/22 (Max Length: 24)
77.137.200.0/21 (

185.198.4.0/24 (Max Length: 24)
185.198.5.0/24 (Max Length: 24)
185.198.6.0/24 (Max Length: 24)
185.198.7.0/24 (Max Length: 24)
185.204.172.0/22 (Max Length: 24)
185.217.28.0/22 (Max Length: 24)
185.223.104.0/22 (Max Length: 24)
185.230.181.0/24 (Max Length: 24)
185.230.182.0/24 (Max Length: 24)
185.230.183.0/24 (Max Length: 24)
185.253.12.0/22 (Max Length: 24)
188.7.0.0/16 (Max Length: 24)
188.28.192.0/18 (Max Length: 18)
188.29.32.0/20 (Max Length: 20)
188.29.48.0/20 (Max Length: 20)
188.30.0.0/22 (Max Length: 22)
188.30.4.0/23 (Max Length: 23)
188.30.6.0/24 (Max Length: 24)
188.30.7.0/25 (Max Length: 25)
188.30.8.0/21 (Max Length: 21)
188.30.16.0/21 (Max Length: 21)
188.30.160.0/19 (Max Length: 19)
188.30.224.0/20 (Max Length: 20)
188.31.0.0/22 (Max Length: 22)
188.31.4.0/23 (Max Length: 23)
188.31.6.0/24 (Max Length: 24)
188.31.32.0/19 (Max Length: 19)
188.141.128.0/17 (Max Length: 24)
188.224.0.0/17 (Max Length: 24)
192.77.114.0/23 (Max Length: 24)
192.112.208.0/24 (Max Length: 24

In [10]:
import lzma
import csv
import requests
from io import BytesIO
from datetime import datetime
from collections import defaultdict

def fetch_roas_and_save_conflicts(date_str: str, target_asn: str):
    tal = "apnic"   
    siblings_map = {
        "AS32787": ["AS35994",  "AS16625",  "AS36183",  "AS12222",  "AS31984",  "AS17204",  "AS26008",  
                    "AS18717",  "AS393234",  "AS393560",  "AS33047",  "AS23454",  "AS36029",  "AS18680",  
                    "AS17334", "AS22207",  "AS16702",  "AS23455",  "AS22452",  "AS30675",  "AS20189",  
                    "AS35993"], # Akamai
        "AS13335": ["AS209242", "AS395747", "AS14789", "AS394536"], # Cloudflare
        "AS19905": ["AS12008",  "AS19911",  "AS397224",  "AS399163",  "AS399156",  "AS397219",  "AS399169",  "AS399170",  
                "AS399164",  "AS397233",  "AS399167",  "AS397229",  "AS397239",  "AS397222",  "AS397235",  "AS397243",  
                "AS399161",  "AS399165",  "AS397232",  "AS397238",  "AS397215",  "AS397223",  "AS399168",  "AS397225",  
                "AS399155",  "AS397218",  "AS397221",  "AS399158",  "AS397237",  "AS399154",  "AS397213",  "AS399159",  
                "AS399153",  "AS397220",  "AS397226",  "AS399160",  "AS399157",  "AS397231",  "AS397227",  "AS397241",  
                "AS397234",  "AS399173",  "AS397240",  "AS397228",  "AS397214",  "AS399177",  "AS397230",  "AS22701", 
                "AS399171",  "AS397216",  "AS397242",  "AS399180",  "AS399162",  "AS399179",  "AS397236",  "AS399176",  
                "AS399172",  "AS399175",  "AS399151",  "AS399166",  "AS397217",  "AS399178",  "AS399152",  "AS399174"
                   ], # Vercara   

        "AS19551": [], # Imperva
        "AS198949": ["AS48851", "AS213232"] # Radware
        "AS35280": ["AS43767"], # F5
        "AS20052": [] # Arbour Network (Netscout) # No prefixes registered in RPKI TALs
        "AS10690" : []
    }
    
    siblings = siblings_map.get(target_asn, [])

    try:
        # Parse date and prepare URL
        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
        url_date = date_obj.strftime('%Y/%m/%d')
        url = f'https://ftp.ripe.net/rpki/{tal}.tal/{url_date}/roas.csv.xz'
        
        print(f"Fetching ROA data from: {url}")
        response = requests.get(url)
        response.raise_for_status()

        # Decompress and read CSV
        with lzma.open(BytesIO(response.content)) as f:
            lines = f.read().decode().splitlines()
            csv_reader = csv.DictReader(lines)
            
            prefix_to_asns = defaultdict(set)

            for row in csv_reader:
                asn = row['ASN'].strip().upper()
                prefix = row['IP Prefix'].strip()
                if asn not in siblings:
                    prefix_to_asns[prefix].add(asn)

        # Filter prefixes for the target ASN
        
        target_prefixes = set()
        for prefix, asns in prefix_to_asns.items():
            if target_asn.upper() in asns:
                target_prefixes.add(prefix)

        
        # Now find how many of those prefixes also have other ASNs
        conflict_prefixes = {prefix: asns for prefix, asns in prefix_to_asns.items() 
                             if prefix in target_prefixes and len(asns) > 1}

        # Save to CSV
        output_file = 'conflicting_roas_'+tal+'_'+target_asn+'.csv'
        with open(output_file, 'w', newline='') as csvfile:
            fieldnames = ['IP Prefix', 'ASNs']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            for prefix, asns in conflict_prefixes.items():
                
                writer.writerow({
                    'IP Prefix': prefix,
                    'ASNs': ', '.join(sorted(asns))
                })

        print(f"\nTotal prefixes registered to {target_asn}: {len(target_prefixes)}")
        print(f"Prefixes also registered to other ASNs: {len(conflict_prefixes)}")
        print(f"Saved conflicting entries to {output_file}")
    except Exception as e:
        print(f"Error: {e}")

# Example usage
if __name__ == "__main__":
#     date_input = input("Enter date (yyyy/mm/dd): ").strip()
    date_input = "2025/06/11"
#     asn_input = input("Enter ASN (e.g., AS12345): ").strip()
    asn_input = "AS13335"
    fetch_roas_and_save_conflicts(date_input, asn_input)


Fetching ROA data from: https://ftp.ripe.net/rpki/apnic.tal/2025/06/11/roas.csv.xz


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [7]:
siblings_map = {
            "32787": ["35994", "16625", "36183", "12222", "31984", "17204", "26008",
                        "18717", "393234", "393560", "33047", "23454", "36029", "18680",
                        "17334", "22207", "16702", "23455", "22452", "30675", "20189",
                        "35993"],  # Akamai
            "13335": ["209242", "395747", "14789", "394536"],  # Cloudflare
            "19905": ["12008", "19911", "397224", "399163", "399156", "397219", "399169", "399170",
                        "399164", "397233", "399167", "397229", "397239", "397222", "397235", "397243",
                        "399161", "399165", "397232", "397238", "397215", "397223", "399168", "397225",
                        "399155", "397218", "397221", "399158", "397237", "399154", "397213", "399159",
                        "399153", "397220", "397226", "399160", "399157", "397231", "397227", "397241",
                        "397234", "399173", "397240", "397228", "397214", "399177", "397230", "22701",
                        "399171", "397216", "397242", "399180", "399162", "399179", "397236", "399176",
                        "399172", "399175", "399151", "399166", "397217", "399178", "399152", "399174"
                        ],  # Vercara

            "19551": [],  # Imperva
            "198949": ["48851", "213232"],  # Radware
            "35280": ["43767"],  # F5
            "20052": [],  # Arbour Network (Netscout) # No prefixes registered in RPKI TALs
            "10690": []
        }
scrubber_asn = "32787"
siblings = siblings_map.get(scrubber_asn, [])
print(siblings)

['35994', '16625', '36183', '12222', '31984', '17204', '26008', '18717', '393234', '393560', '33047', '23454', '36029', '18680', '17334', '22207', '16702', '23455', '22452', '30675', '20189', '35993']
